
# 05b — Gemma Full-Context Validation (Analysis Only)

This notebook analyzes the **76 reconstructed full-context Gemma records** produced by `05a_Gemma_Full_Context_Raw_Extraction.ipynb`.

It does **not** load Gemma or rerun model forward passes. It loads the 76 saved full-context tensor archives, recomputes the frozen H1–H4 features using the same definitions as Gemma Phase 04a, evaluates the frozen classifier, runs permutation tests, compares against completion-only Phase 04a, checks repeated grouped-CV stability, and saves a compact result bundle.


## 1. Imports and configuration

In [6]:

import json, platform, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
N_SPLITS = 5
CLASSIFIER_C = 0.01
N_HEAD_PERMUTATIONS = 2000
N_H4_PERMUTATIONS = 10000
N_STABILITY_SEEDS = 50
np.random.seed(SEED)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)


Python: 3.12.13
NumPy: 2.0.2


## 2. Resolve Phase 05a and Phase 04a inputs

In [7]:
from pathlib import Path

# ============================================================
# INPUTS
# ============================================================

FULL_CONTEXT_DIR = Path(
    "/kaggle/input/notebooks/gunjalgupta/"
    "gemma-reconstructed/full_context_gemma"
)

COMPLETION_ONLY_FEATURES_PATH = Path(
    "/kaggle/input/notebooks/gunjalgupta/"
    "phase-4a-gemma/pivot_research_gemma/"
    "phase4_gemma_features.npz"
)

# ============================================================
# OUTPUTS
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/full_context_gemma_analysis"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FEATURES_OUT = (
    OUTPUT_DIR
    / "phase4_gemma_full_context_features.npz"
)

RESULTS_OUT = (
    OUTPUT_DIR
    / "full_context_gemma_results.json"
)

COMPARISON_OUT = (
    OUTPUT_DIR
    / "full_context_vs_completion_only_comparison.csv"
)

STABILITY_OUT = (
    OUTPUT_DIR
    / "full_context_grouped_cv_stability.csv"
)

SUMMARY_OUT = (
    OUTPUT_DIR
    / "full_context_gemma_summary.txt"
)

# ============================================================
# FILES PRODUCED BY GEMMA 05a
# ============================================================

RECORD_DIR = FULL_CONTEXT_DIR / "records"

META_PATH = (
    FULL_CONTEXT_DIR
    / "full_context_gemma_metadata.jsonl"
)

MANIFEST_PATH = (
    FULL_CONTEXT_DIR
    / "full_context_gemma_manifest.json"
)

ALIGNMENT_PATH = (
    FULL_CONTEXT_DIR
    / "full_context_gemma_alignment_audit.csv"
)

# ============================================================
# VALIDATION
# ============================================================

required_paths = [
    FULL_CONTEXT_DIR,
    RECORD_DIR,
    META_PATH,
    MANIFEST_PATH,
    COMPLETION_ONLY_FEATURES_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input not found: {path}"
        )

record_files = sorted(
    RECORD_DIR.glob("record_*.npz")
)

if len(record_files) != 76:
    raise RuntimeError(
        f"Expected 76 record files, "
        f"found {len(record_files)} in {RECORD_DIR}"
    )

print("Full-context directory:")
print(FULL_CONTEXT_DIR)

print("\nRecord directory:")
print(RECORD_DIR)

print("\nRecord files found:")
print(len(record_files))

print("\nMetadata:")
print(META_PATH)

print("\nManifest:")
print(MANIFEST_PATH)

print("\nAlignment audit:")
print(
    ALIGNMENT_PATH
    if ALIGNMENT_PATH.exists()
    else "Not found — optional"
)

print("\nCompletion-only features:")
print(COMPLETION_ONLY_FEATURES_PATH)

print("\nOutput directory:")
print(OUTPUT_DIR)

Full-context directory:
/kaggle/input/notebooks/gunjalgupta/gemma-reconstructed/full_context_gemma

Record directory:
/kaggle/input/notebooks/gunjalgupta/gemma-reconstructed/full_context_gemma/records

Record files found:
76

Metadata:
/kaggle/input/notebooks/gunjalgupta/gemma-reconstructed/full_context_gemma/full_context_gemma_metadata.jsonl

Manifest:
/kaggle/input/notebooks/gunjalgupta/gemma-reconstructed/full_context_gemma/full_context_gemma_manifest.json

Alignment audit:
/kaggle/input/notebooks/gunjalgupta/gemma-reconstructed/full_context_gemma/full_context_gemma_alignment_audit.csv

Completion-only features:
/kaggle/input/notebooks/gunjalgupta/phase-4a-gemma/pivot_research_gemma/phase4_gemma_features.npz

Output directory:
/kaggle/working/full_context_gemma_analysis


## 3. Load and verify Phase 05a metadata

In [10]:
def load_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


metadata = load_jsonl(META_PATH)

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

# ------------------------------------------------------------
# Infer architecture directly from the first saved tensor file
# ------------------------------------------------------------

first_record_path = RECORD_DIR / metadata[0]["record_file"]

if not first_record_path.exists():
    raise FileNotFoundError(first_record_path)

with np.load(first_record_path, allow_pickle=False) as archive:
    first_weights = archive["weights"]
    first_outputs = archive["outputs"]

# weights: [layers, heads, queries, keys]
N_LAYERS, N_HEADS, N_QUERIES, MAX_KEYS = first_weights.shape

# outputs: [layers, queries, heads, d_head]
z_layers, z_queries, z_heads, D_HEAD = first_outputs.shape

if z_layers != N_LAYERS:
    raise RuntimeError(
        f"Layer mismatch: weights={N_LAYERS}, outputs={z_layers}"
    )

if z_queries != N_QUERIES:
    raise RuntimeError(
        f"Query mismatch: weights={N_QUERIES}, outputs={z_queries}"
    )

if z_heads != N_HEADS:
    raise RuntimeError(
        f"Head mismatch: weights={N_HEADS}, outputs={z_heads}"
    )

# Verify inferred dimensions against manifest fields that do exist.
if int(manifest["query_window_size"]) != N_QUERIES:
    raise RuntimeError(
        "Manifest query_window_size does not match saved tensors."
    )

if int(manifest["max_key_context"]) != MAX_KEYS:
    raise RuntimeError(
        "Manifest max_key_context does not match saved tensors."
    )

EARLY_KEY_RANGE = 50

labels = np.array(
    [row["label"] for row in metadata],
    dtype=str,
)

categories = np.array(
    [row.get("category", "unknown") for row in metadata],
    dtype=str,
)

prompt_ids = np.array(
    [row["prompt_id"] for row in metadata],
    dtype=str,
)

record_keys = np.array(
    [
        f"{row['prompt_id']}#{row.get('sample_idx', '?')}"
        for row in metadata
    ],
    dtype=str,
)

y = (
    labels == "correction"
).astype(np.int8)

print("Records:", len(metadata))
print("Labels:", dict(Counter(labels)))
print("Categories:", dict(Counter(categories)))
print("Unique prompts:", len(np.unique(prompt_ids)))

print(
    "Inferred architecture:",
    {
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "n_queries": N_QUERIES,
        "max_keys": MAX_KEYS,
        "d_head": D_HEAD,
    },
)

print(
    "Manifest extraction status:",
    {
        "n_requested": manifest["n_requested"],
        "n_extracted": manifest["n_extracted"],
        "n_errors": manifest["n_errors"],
        "prompt_truncation_count": manifest[
            "prompt_truncation_count"
        ],
        "completion_length_mismatch_count": manifest[
            "completion_length_mismatch_count"
        ],
    },
)

assert len(metadata) == 76
assert int(y.sum()) == 13
assert int((y == 0).sum()) == 63

assert manifest["study_condition"] == (
    "reconstructed_full_context"
)
assert int(manifest["n_extracted"]) == 76
assert int(manifest["n_errors"]) == 0
assert int(manifest["prompt_truncation_count"]) == 0
assert (
    int(
        manifest[
            "completion_length_mismatch_count"
        ]
    )
    == 0
)

assert (
    N_LAYERS,
    N_HEADS,
    N_QUERIES,
    MAX_KEYS,
    D_HEAD,
) == (
    26,
    8,
    30,
    250,
    256,
)

print("Metadata and architecture validation passed.")

Records: 76
Labels: {np.str_('hallucination_pivot'): 63, np.str_('correction'): 13}
Categories: {np.str_('misdirection'): 32, np.str_('math'): 35, np.str_('logic'): 9}
Unique prompts: 47
Inferred architecture: {'n_layers': 26, 'n_heads': 8, 'n_queries': 30, 'max_keys': 250, 'd_head': 256}
Manifest extraction status: {'n_requested': 76, 'n_extracted': 76, 'n_errors': 0, 'prompt_truncation_count': 0, 'completion_length_mismatch_count': 0}
Metadata and architecture validation passed.


## 4. Load the 76 full-context tensor archives

In [11]:

weights_list=[]; outputs_list=[]
expected_w=(N_LAYERS,N_HEADS,N_QUERIES,MAX_KEYS); expected_z=(N_LAYERS,N_QUERIES,N_HEADS,D_HEAD)
for row in tqdm(metadata,desc='Loading full-context tensors'):
    p=RECORD_DIR/row['record_file']
    if not p.exists(): raise FileNotFoundError(p)
    with np.load(p,allow_pickle=False) as a:
        w=a['weights'].astype(np.float32); z=a['outputs'].astype(np.float32)
    if w.shape!=expected_w: raise RuntimeError(f'{p.name}: {w.shape} != {expected_w}')
    if z.shape!=expected_z: raise RuntimeError(f'{p.name}: {z.shape} != {expected_z}')
    if not np.isfinite(w).all() or not np.isfinite(z).all(): raise RuntimeError(f'Non-finite tensors in {p.name}')
    weights_list.append(w); outputs_list.append(z)
weights=np.stack(weights_list); outputs=np.stack(outputs_list)
valid_key_starts=np.array([int(r['valid_key_start']) for r in metadata])
print('Weights:',weights.shape); print('Outputs:',outputs.shape)
assert weights.shape==(76,26,8,30,250) and outputs.shape==(76,26,30,8,256)


Loading full-context tensors:   0%|          | 0/76 [00:00<?, ?it/s]

Weights: (76, 26, 8, 30, 250)
Outputs: (76, 26, 30, 8, 256)


## 5. Compute frozen H1–H4 features

Definitions match Gemma Phase 04a exactly: H1 early retained-context attention mass, H2 per-head output magnitude, H3 mask-defined normalized entropy, and H4 layers 17–25 divided by layers 0–7.

In [12]:

n_records=len(metadata)
h1=np.zeros((n_records,N_LAYERS,N_HEADS),dtype=np.float32)
for i in range(n_records):
    v=int(valid_key_starts[i]); e=min(v+EARLY_KEY_RANGE,MAX_KEYS)
    h1[i]=weights[i,:,:,:,v:e].sum(axis=-1).mean(axis=-1)
h2=np.linalg.norm(outputs,axis=-1).mean(axis=2).astype(np.float32)

def normalized_mask_entropy(row, valid_start, query_index):
    real=row[valid_start:].astype(np.float64); total=len(real)
    available=max(1,min(total-(N_QUERIES-1-query_index),total))
    p=real[:available]; mass=p.sum()
    if available<=1 or mass<=0: return 0.0
    p=p/mass; nz=p>0
    return float(-np.sum(p[nz]*np.log(p[nz]))/np.log(available))

h3=np.zeros((n_records,N_LAYERS,N_HEADS),dtype=np.float32)
for i in tqdm(range(n_records),desc='Computing H3'):
    v=int(valid_key_starts[i])
    for L in range(N_LAYERS):
        for H in range(N_HEADS):
            h3[i,L,H]=np.mean([normalized_mask_entropy(weights[i,L,H,q,:],v,q) for q in range(N_QUERIES)])
EARLY_LAYERS=list(range(0,N_LAYERS//3)); LATE_LAYERS=list(range(2*N_LAYERS//3,N_LAYERS))
assert EARLY_LAYERS==list(range(0,8)) and LATE_LAYERS==list(range(17,26))
early_mag=h2[:,EARLY_LAYERS,:].mean(axis=(1,2)); late_mag=h2[:,LATE_LAYERS,:].mean(axis=(1,2))
h4=(late_mag/(early_mag+1e-10)).astype(np.float32)
for arr,shape in [(h1,(76,26,8)),(h2,(76,26,8)),(h3,(76,26,8)),(h4,(76,))]:
    assert arr.shape==shape and np.isfinite(arr).all()
print('H1/H2/H3/H4:',h1.shape,h2.shape,h3.shape,h4.shape)


Computing H3:   0%|          | 0/76 [00:00<?, ?it/s]

H1/H2/H3/H4: (76, 26, 8) (76, 26, 8) (76, 26, 8) (76,)


## 6. Build and save the 625-feature archive

In [13]:

X=np.concatenate([h1.reshape(n_records,-1),h2.reshape(n_records,-1),h3.reshape(n_records,-1),h4.reshape(n_records,1)],axis=1).astype(np.float32)
feature_names=np.array(
 [f'H1_L{L:02d}_H{H:02d}' for L in range(N_LAYERS) for H in range(N_HEADS)] +
 [f'H2_L{L:02d}_H{H:02d}' for L in range(N_LAYERS) for H in range(N_HEADS)] +
 [f'H3_L{L:02d}_H{H:02d}' for L in range(N_LAYERS) for H in range(N_HEADS)] + ['H4_late_early_ratio'],dtype=str)
assert X.shape==(76,625) and len(feature_names)==625 and np.isfinite(X).all()
np.savez_compressed(FEATURES_OUT,X=X,y=y,prompt_ids=prompt_ids,categories=categories,record_keys=record_keys,feature_names=feature_names,h1=h1,h2=h2,h3=h3,h4=h4,n_layers=np.array(N_LAYERS),n_heads=np.array(N_HEADS),d_head=np.array(D_HEAD),query_window=np.array(N_QUERIES),max_key_context=np.array(MAX_KEYS),early_layers=np.array(EARLY_LAYERS),late_layers=np.array(LATE_LAYERS),replay_condition=np.array('reconstructed_full_context'))
print('Saved:',FEATURES_OUT); print('Shape:',X.shape)


Saved: /kaggle/working/full_context_gemma_analysis/phase4_gemma_full_context_features.npz
Shape: (76, 625)


## 7. Frozen classifier evaluation

In [14]:

classifier=Pipeline([('scale',StandardScaler()),('logreg',LogisticRegression(C=CLASSIFIER_C,penalty='l2',class_weight='balanced',solver='liblinear',max_iter=5000,random_state=SEED))])
def evaluate_oof(Xe,ye,mode,groups=None,seed=42):
    if mode=='instance': splits=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=seed).split(Xe,ye)
    elif mode=='grouped': splits=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=seed).split(Xe,ye,groups=groups)
    else: raise ValueError(mode)
    proba=np.full(len(ye),np.nan); pred=np.full(len(ye),-1); fold_ids=np.full(len(ye),-1)
    for fold,(tr,te) in enumerate(splits):
        if len(np.unique(ye[tr]))<2: raise RuntimeError(f'{mode} seed {seed} fold {fold}: one-class training split')
        fit=clone(classifier).fit(Xe[tr],ye[tr]); proba[te]=fit.predict_proba(Xe[te])[:,1]; pred[te]=fit.predict(Xe[te]); fold_ids[te]=fold
    if np.isnan(proba).any(): raise RuntimeError('Unassigned predictions')
    return {'auc':float(roc_auc_score(ye,proba)),'balanced_accuracy':float(balanced_accuracy_score(ye,pred)),'f1_correction':float(f1_score(ye,pred,pos_label=1,zero_division=0)),'average_precision':float(average_precision_score(ye,proba)),'probabilities':proba,'predictions':pred,'fold_ids':fold_ids}
instance_result=evaluate_oof(X,y,'instance',seed=SEED)
grouped_result=evaluate_oof(X,y,'grouped',groups=prompt_ids,seed=SEED)
print('Instance:',{k:round(v,4) for k,v in instance_result.items() if np.isscalar(v)})
print('Grouped:',{k:round(v,4) for k,v in grouped_result.items() if np.isscalar(v)})


Instance: {'auc': 0.5397, 'balanced_accuracy': 0.5153, 'f1_correction': 0.2692, 'average_precision': 0.2362}
Grouped: {'auc': 0.6142, 'balanced_accuracy': 0.5696, 'f1_correction': 0.3137, 'average_precision': 0.2332}


## 8. H1–H4 permutation tests

In [15]:

def permutation_pvalues_matrix(matrix,labels_binary,n_perm,seed):
    rng=np.random.default_rng(seed)
    observed=matrix[labels_binary==1].mean(axis=0)-matrix[labels_binary==0].mean(axis=0)
    exceed=np.zeros(matrix.shape[1],dtype=np.int64)
    for _ in range(n_perm):
        p=rng.permutation(labels_binary); d=matrix[p==1].mean(axis=0)-matrix[p==0].mean(axis=0); exceed+=(np.abs(d)>=np.abs(observed))
    pv=(exceed+1)/(n_perm+1); q=multipletests(pv,alpha=.05,method='fdr_bh')[1]
    return observed,pv,q
family_slices={'H1':slice(0,208),'H2':slice(208,416),'H3':slice(416,624)}; hypothesis_results={}
for offset,(family,sl) in enumerate(family_slices.items()):
    observed,pv,q=permutation_pvalues_matrix(X[:,sl],y,N_HEAD_PERMUTATIONS,SEED+offset); b=int(np.argmin(pv))
    hypothesis_results[family]={'n_fdr_significant':int((q<.05).sum()),'best_raw_p':float(pv.min()),'best_adjusted_p':float(q.min()),'best_feature':str(feature_names[sl][b]),'best_effect_correction_minus_hallucination':float(observed[b])}
    print(family,hypothesis_results[family])
rng=np.random.default_rng(SEED); h4_obs=float(h4[y==1].mean()-h4[y==0].mean()); exceed=0
for _ in range(N_H4_PERMUTATIONS):
    p=rng.permutation(y); d=float(h4[p==1].mean()-h4[p==0].mean()); exceed+=abs(d)>=abs(h4_obs)
h4_p=(exceed+1)/(N_H4_PERMUTATIONS+1)
hypothesis_results['H4']={'correction_minus_hallucination':h4_obs,'hallucination_minus_correction':-h4_obs,'p_two_sided':float(h4_p),'correction_mean':float(h4[y==1].mean()),'hallucination_mean':float(h4[y==0].mean()),'early_layers':EARLY_LAYERS,'late_layers':LATE_LAYERS}
print('H4:',hypothesis_results['H4'])


H1 {'n_fdr_significant': 0, 'best_raw_p': 0.2108945527236382, 'best_adjusted_p': 0.9064267866066966, 'best_feature': 'H1_L00_H02', 'best_effect_correction_minus_hallucination': 0.03551151603460312}
H2 {'n_fdr_significant': 0, 'best_raw_p': 0.0014992503748125937, 'best_adjusted_p': 0.3118440779610195, 'best_feature': 'H2_L04_H00', 'best_effect_correction_minus_hallucination': -0.48270297050476074}
H3 {'n_fdr_significant': 0, 'best_raw_p': 0.009995002498750625, 'best_adjusted_p': 0.9877648240556339, 'best_feature': 'H3_L00_H05', 'best_effect_correction_minus_hallucination': -0.05257517099380493}
H4: {'correction_minus_hallucination': -0.004859864711761475, 'hallucination_minus_correction': 0.004859864711761475, 'p_two_sided': 0.5205479452054794, 'correction_mean': 0.694621205329895, 'hallucination_mean': 0.6994810700416565, 'early_layers': [0, 1, 2, 3, 4, 5, 6, 7], 'late_layers': [17, 18, 19, 20, 21, 22, 23, 24, 25]}


## 9. Paired comparison with completion-only Phase 04a

In [16]:

def safe_featurewise_correlations(A,B):
    r=np.full(A.shape[1],np.nan)
    for j in range(A.shape[1]):
        if np.std(A[:,j])>0 and np.std(B[:,j])>0: r[j]=pearsonr(A[:,j],B[:,j]).statistic
    return r
with np.load(COMPLETION_ONLY_FEATURES_PATH,allow_pickle=False) as old:
    X_old=old['X'].astype(np.float64); y_old=old['y'].astype(int); prompts_old=old['prompt_ids'].astype(str); categories_old=old['categories'].astype(str); keys_old=old['record_keys'].astype(str) if 'record_keys' in old.files else None
if keys_old is not None:
    idx={k:i for i,k in enumerate(keys_old)}
    if set(record_keys)!=set(keys_old): raise RuntimeError('Completion-only and full-context record keys differ')
    order=np.array([idx[k] for k in record_keys]); X_old=X_old[order]; y_old=y_old[order]; prompts_old=prompts_old[order]; categories_old=categories_old[order]
else:
    assert X_old.shape==X.shape and np.array_equal(y_old,y) and np.array_equal(prompts_old,prompt_ids)
comparison_rows=[]
for family,sl in {**family_slices,'H4':slice(624,625)}.items():
    r=safe_featurewise_correlations(X_old[:,sl],X[:,sl]); a=X_old[:,sl].ravel(); b=X[:,sl].ravel(); entry=float(pearsonr(a,b).statistic) if np.std(a)>0 and np.std(b)>0 else float('nan')
    comparison_rows.append({'family':family,'n_features':sl.stop-sl.start,'median_featurewise_r':float(np.nanmedian(r)),'mean_featurewise_r':float(np.nanmean(r)),'entrywise_r':entry,'median_absolute_change':float(np.median(np.abs(X[:,sl]-X_old[:,sl])))})
comparison_df=pd.DataFrame(comparison_rows); comparison_df.to_csv(COMPARISON_OUT,index=False); display(comparison_df)
old_instance=evaluate_oof(X_old,y,'instance',seed=SEED); old_grouped=evaluate_oof(X_old,y,'grouped',groups=prompt_ids,seed=SEED)
paired_comparison={'completion_only_instance_auc':old_instance['auc'],'full_context_instance_auc':instance_result['auc'],'instance_auc_difference':instance_result['auc']-old_instance['auc'],'completion_only_grouped_auc':old_grouped['auc'],'full_context_grouped_auc':grouped_result['auc'],'grouped_auc_difference':grouped_result['auc']-old_grouped['auc'],'feature_family_correlations':comparison_rows}
print(json.dumps(paired_comparison,indent=2))


,family,n_features,median_featurewise_r,mean_featurewise_r,entrywise_r,median_absolute_change
0,H1,208,0.611585,0.592264,0.545551,0.264575
1,H2,208,0.334719,0.337711,0.649889,3.188706
2,H3,208,0.163687,0.142661,0.712379,0.079690
3,H4,1,0.195453,0.195453,0.195453,0.247156


{
  "completion_only_instance_auc": 0.49938949938949945,
  "full_context_instance_auc": 0.5396825396825398,
  "instance_auc_difference": 0.040293040293040316,
  "completion_only_grouped_auc": 0.63003663003663,
  "full_context_grouped_auc": 0.6141636141636142,
  "grouped_auc_difference": -0.015873015873015817,
  "feature_family_correlations": [
    {
      "family": "H1",
      "n_features": 208,
      "median_featurewise_r": 0.6115845710964185,
      "mean_featurewise_r": 0.5922639168788372,
      "entrywise_r": 0.5455510804665299,
      "median_absolute_change": 0.26457517920061946
    },
    {
      "family": "H2",
      "n_features": 208,
      "median_featurewise_r": 0.33471926839145916,
      "mean_featurewise_r": 0.3377106650815126,
      "entrywise_r": 0.6498887296722393,
      "median_absolute_change": 3.1887062788009644
    },
    {
      "family": "H3",
      "n_features": 208,
      "median_featurewise_r": 0.16368694272980677,
      "mean_featurewise_r": 0.1426614614800445,


## 10. Repeated grouped-CV stability

In [17]:

stability_rows=[]; failures=[]
for seed in tqdm(range(N_STABILITY_SEEDS),desc='Repeated grouped CV'):
    try:
        r=evaluate_oof(X,y,'grouped',groups=prompt_ids,seed=seed)
        stability_rows.append({'seed':seed,'auc':r['auc'],'balanced_accuracy':r['balanced_accuracy'],'f1_correction':r['f1_correction'],'average_precision':r['average_precision']})
    except Exception as e: failures.append({'seed':seed,'error_type':type(e).__name__,'error':str(e)})
stability_df=pd.DataFrame(stability_rows); stability_df.to_csv(STABILITY_OUT,index=False)
if stability_df.empty: raise RuntimeError('All grouped-CV runs failed')
a=stability_df['auc'].to_numpy(); b=stability_df['balanced_accuracy'].to_numpy()
stability_summary={'requested_seeds':N_STABILITY_SEEDS,'successful_seeds':len(stability_df),'failed_seeds':len(failures),'auc_mean':float(a.mean()),'auc_median':float(np.median(a)),'auc_sd':float(a.std()),'auc_min':float(a.min()),'auc_max':float(a.max()),'auc_95_percentile_interval':[float(np.percentile(a,2.5)),float(np.percentile(a,97.5))],'proportion_auc_above_0_5':float(np.mean(a>.5)),'balanced_accuracy_mean':float(b.mean()),'balanced_accuracy_median':float(np.median(b)),'balanced_accuracy_95_percentile_interval':[float(np.percentile(b,2.5)),float(np.percentile(b,97.5))],'failures':failures}
print(json.dumps(stability_summary,indent=2))


Repeated grouped CV:   0%|          | 0/50 [00:00<?, ?it/s]

{
  "requested_seeds": 50,
  "successful_seeds": 50,
  "failed_seeds": 0,
  "auc_mean": 0.5595360195360196,
  "auc_median": 0.561050061050061,
  "auc_sd": 0.04799936415083032,
  "auc_min": 0.463980463980464,
  "auc_max": 0.6556776556776557,
  "auc_95_percentile_interval": [
    0.47402319902319906,
    0.6492979242979243
  ],
  "proportion_auc_above_0_5": 0.9,
  "balanced_accuracy_mean": 0.521013431013431,
  "balanced_accuracy_median": 0.5311355311355311,
  "balanced_accuracy_95_percentile_interval": [
    0.42933455433455436,
    0.5888888888888889
  ],
  "failures": []
}


## 11. Save final results

In [20]:
results = {
    "condition": "reconstructed_full_context",
    "analysis_source": "Gemma 05a raw full-context tensors",
    "n_records": int(len(y)),
    "n_corrections": int(y.sum()),
    "n_hallucination_pivots": int((y == 0).sum()),
    "n_unique_prompts": int(len(np.unique(prompt_ids))),
    "feature_shape": list(X.shape),
    "architecture": {
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "d_head": D_HEAD,
        "query_window": N_QUERIES,
        "max_key_context": MAX_KEYS,
        "early_layers": EARLY_LAYERS,
        "late_layers": LATE_LAYERS,
    },
    "classifier": {
        "pipeline": "StandardScaler + class-balanced L2 logistic regression",
        "C": CLASSIFIER_C,
        "seed": SEED,
        "instance": {
            k: v for k, v in instance_result.items()
            if np.isscalar(v)
        },
        "grouped": {
            k: v for k, v in grouped_result.items()
            if np.isscalar(v)
        },
    },
    "hypothesis_tests": hypothesis_results,
    "paired_completion_only_comparison": paired_comparison,
    "grouped_cv_stability": stability_summary,
}

with RESULTS_OUT.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

lines = [
    "=" * 78,
    "GEMMA FULL-CONTEXT VALIDATION — FINAL SUMMARY",
    "=" * 78,
    f"Records: {len(y)} ({int(y.sum())} correction / {int((y==0).sum())} hallucination)",
    f"Instance AUC / BA: {instance_result['auc']:.3f} / {instance_result['balanced_accuracy']:.3f}",
    f"Grouped  AUC / BA: {grouped_result['auc']:.3f} / {grouped_result['balanced_accuracy']:.3f}",
    (
        f"Repeated grouped AUC mean / median / 95% interval: "
        f"{stability_summary['auc_mean']:.3f} / "
        f"{stability_summary['auc_median']:.3f} / "
        f"[{stability_summary['auc_95_percentile_interval'][0]:.3f}, "
        f"{stability_summary['auc_95_percentile_interval'][1]:.3f}]"
    ),
    f"H1: {hypothesis_results['H1']['n_fdr_significant']}/208 FDR-significant heads",
    f"H2: {hypothesis_results['H2']['n_fdr_significant']}/208 FDR-significant heads",
    f"H3: {hypothesis_results['H3']['n_fdr_significant']}/208 FDR-significant heads",
    (
        f"H4 hallucination - correction: "
        f"{hypothesis_results['H4']['hallucination_minus_correction']:+.4f}; "
        f"p={hypothesis_results['H4']['p_two_sided']:.4f}"
    ),
    (
        f"Completion-only -> full-context grouped AUC: "
        f"{paired_comparison['completion_only_grouped_auc']:.3f} -> "
        f"{paired_comparison['full_context_grouped_auc']:.3f}"
    ),
    "",
    f"Saved results: {RESULTS_OUT}",
]

summary = "\n".join(lines)

with SUMMARY_OUT.open("w", encoding="utf-8") as f:
    f.write(summary)

print(summary)

GEMMA FULL-CONTEXT VALIDATION — FINAL SUMMARY
Records: 76 (13 correction / 63 hallucination)
Instance AUC / BA: 0.540 / 0.515
Grouped  AUC / BA: 0.614 / 0.570
Repeated grouped AUC mean / median / 95% interval: 0.560 / 0.561 / [0.474, 0.649]
H1: 0/208 FDR-significant heads
H2: 0/208 FDR-significant heads
H3: 0/208 FDR-significant heads
H4 hallucination - correction: +0.0049; p=0.5205
Completion-only -> full-context grouped AUC: 0.630 -> 0.614

Saved results: /kaggle/working/full_context_gemma_analysis/full_context_gemma_results.json


## 12. Create compact output bundle

In [21]:

import shutil
bundle=shutil.make_archive('/kaggle/working/Gemma_05b_Full_Context_Validation_Outputs','zip',root_dir=OUTPUT_DIR)
print('Bundle:',bundle)


Bundle: /kaggle/working/Gemma_05b_Full_Context_Validation_Outputs.zip
